# Ngày 5 · SGD, mini-batch và chuẩn hóa

- So sánh batch, stochastic và mini-batch
- Quan sát ảnh hưởng của thang đo đặc trưng

**Đọc:**
- UC Berkeley CS189, Stochastic Gradient Descent
- Stanford CS229, §1.1–1.2

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge

DATA = next((p for p in (Path('data'), Path('../data')) if (p/'real_estate.csv').is_file()), None)
if DATA is None:
    raise FileNotFoundError('Mở notebook mẫu từ thư mục py30 hoặc bai_lam.')
real_estate = pd.read_csv(DATA / 'real_estate.csv')

def _check(name, fn):
    try:
        ok, detail = fn()
    except NotImplementedError:
        ok, detail = False, 'Chưa cài đặt (NotImplementedError)'
    except Exception as error:
        ok, detail = False, f'{type(error).__name__}: {error}'
    return {'name': name, 'ok': bool(ok), 'detail': str(detail)}

def _report(ex_id, *checks):
    print('__PY30__' + json.dumps({'id': ex_id, 'checks': list(checks)}, ensure_ascii=False))
def _batch_check(fn):
    pieces=list(fn(11,4,np.random.default_rng(3)))
    return list(map(len,pieces))==[4,4,3] and np.array_equal(np.sort(np.concatenate(pieces)),np.arange(11))
def _sgd_check(fn):
    x=np.log1p(real_estate.mrt_distance_m.to_numpy())
    X=np.column_stack([np.ones(len(x)),(x-x.mean())/x.std()])
    y=real_estate.price_10k_ntd_per_ping.to_numpy()
    w=fn(X,y,np.zeros(2),.02,60,16,42)
    opt=np.linalg.lstsq(X,y,rcond=None)[0]
    return np.mean((X@w-y)**2)<=1.05*np.mean((X@opt-y)**2)

def _reshuffles(fn):
    rng = np.random.default_rng(5)
    e1 = np.concatenate(list(fn(20, 6, rng))); e2 = np.concatenate(list(fn(20, 6, rng)))
    return not np.array_equal(e1, np.arange(20)) and not np.array_equal(e1, e2)
def _stochastic(fn):
    x=np.log1p(real_estate.mrt_distance_m.to_numpy()); X=np.column_stack([np.ones(len(x)),(x-x.mean())/x.std()])
    y=real_estate.price_10k_ntd_per_ping.to_numpy()
    a=fn(X,y,np.zeros(2),.05,1,8,1); b=fn(X,y,np.zeros(2),.05,1,8,2)
    return not np.allclose(a,b)


## Một epoch mini-batch

Gợi ý: Gọi rng.permutation(n) ở đầu mỗi epoch; trong sgd tạo rng = np.random.default_rng(seed) một lần.

In [ ]:
def iterate_minibatches(n,batch_size,rng):
    raise NotImplementedError

def sgd(X,y,w,lr,epochs,batch_size,seed):
    raise NotImplementedError


In [ ]:
_report('d5-batches',
    _check('Mỗi mẫu xuất hiện đúng một lần', lambda: (_batch_check(iterate_minibatches), '11 mẫu với batch size 4.')),
    _check('Loss không vượt 105% tối ưu', lambda: (_sgd_check(sgd), 'Đối chiếu trên đặc trưng UCI đã chuẩn hóa.')),
    _check('Xáo trộn lại ở mỗi epoch', lambda: (_reshuffles(iterate_minibatches), 'Hai epoch liên tiếp có thứ tự khác nhau.')),
    _check('SGD phụ thuộc seed (không phải batch GD)', lambda: (_stochastic(sgd), 'Hai seed khác nhau cho w khác nhau sau 1 epoch.')),
)


## Dự án

Trong House Prices, chọn các cột số khác thang đo. Dùng median, mean và độ lệch chuẩn của train rồi so loss theo số mẫu đã xem.